In [2]:
import numpy as np
from numpy import random
import itertools
from scipy import stats

In [46]:
class NK_model(object):
    
    def __init__(self, n, k):
        self.n = n
        self.k = k
        self.start_type = 0    # the initial given start genotype
        self.get_epistic_site()  # get the epistic network among the sites (including the site itself) 
        
        self.allelic_each_site = [[] for _ in range(self.n)]   # will be used to store the episitc combinations at each site
        self.fitness_list = [{} for _ in range(self.n)]   # will be used to store the fitness of epistic combinations at each
                                                          # site
        self.genotype_list = []   # will be used to store the genotypes generated (subset of all the genotypes)
        self.geno_fitness = []   # will be used to store the calculated fitness of genotypes generated
        self.landscape = {}  # will be used to store the generated landscape (subset of the whole landscape)
        
        self.repeat_adaptive_walk()
        self.search_different_allele()
        self.creat_landscape_subset()
        
        
    @staticmethod
    def generate_start_type(seq_length):
        '''generate a genotype as the start_type
        
        parameters: 
        ----------
            seq_length: int, the length of the wanted genotype sequence
        output: 
        -------
            str that represents the start_type
        '''
        
        a_list = []
        for i in range(seq_length):  # generate a list randomly composed of 0 and 1
            a_list.append(random.randint(0, 2))
            
        start_type_list = []   
        for i in a_list:   # convert the integer elements (0 and 1) in a_list into string elements ('0' and '1') and append 
            start_type_list.append(str(i))   # into start_type_list
            
        start_type = ''.join(start_type_list)  # join the elements in start_type_list into a string
        
        return start_type
    
    
    @staticmethod
    def creat_neighbor(genotype, mutant_num):
        '''creat the mutant_num-mutant neighbors of the given genotype
        
        parameters: 
        ----------
            genotype: str that repesents genotype, eg: "01", "00"
            mutant_num: int, the number of mutant site to creat neighbors
            
        output: 
        -------
            list that store the mutant_num-mutant neighbors of the given genotype
        '''
        
        total_neigh_list= []  # will be used to store the created neighbors of the given genotype
    
        geno_list=list(genotype) # convert the given genotype (str) into a list
    
        indices = list(range(len(geno_list))) # get all the indices for the given genotype
    
        mutant_indices = list(itertools.combinations(indices, mutant_num)) # get all possible combinations of picking 
                                                                    # mutant_num intergers from indices (i.e. mutant sites)
    
        for i in mutant_indices:  # get all its mutant_num -mutant neighbors of the given genotype
            neigh_list = geno_list.copy()
            for j in i:
                if geno_list[j] == '0':
                    neigh_list[j] = '1'
                else:
                    neigh_list[j] = '0'
            
            total_neigh_list.append(''.join(neigh_list))
    
        return total_neigh_list
    
    
    def get_epistic_site(self):
        '''Generate the epistic networks among the sites'''

        self.epistic_site = []
        for i in range(self.n):
            all_site = [x for x in range(self.n) if x != i]  # get the indexes of all sites excluding i
            
            indices = set()   # will be used to store the indexes of epistic sites
            while len(indices) < self.k:
                indices.add(random.randint(0, len(all_site))) # randomly pick the indexes of k elements from all_site

            indices = list(indices)  # convert the set into list
            
            one_site = [i]   # first append the site under study into one_site (a little bit different from 0615 code)
            for j in indices:  # get the epistic sites of site i
                one_site.append(all_site[j])
            
            self.epistic_site.append(one_site) 
           
        
    @staticmethod     
    def get_epistic_combination(genotype, network):
        '''generate the epistic pairs in a given genotype according to the network
        
        parameters: 
        ----------
            genotype: str that repesents a genotype, 
            network: nested list that stores the epistic networks among the sites (self.epistic_site)
            
        output: 
        -------
            list that stores all the epistic combinations in the given genotype
        '''
        
        epistic_combination = []   # will be used to store the epistic combinations in the given genotype
        
        for j in range(len(genotype)):
            one_site = []   
            for i in network[j]:   # get all epistic alleles in one site (store in list one_site)
                one_site.append(genotype[i])
                
            epistic_combination.append(''.join(one_site))  # combine the epistic alleles and store into epistic_combination
                
        return epistic_combination
        
        
    @staticmethod      
    def cal_fitness(epistic_pair, fitness_list):
        '''calculate the fitness of a given genotype according to the fitness_list (here the epistic_pair is generate by
        the genotype, i.e. the epistic_combination in get_epistic_combination(genotype, network) method)
        
        parameters: 
        ----------
            epistic_pair: list that stores the epistic combinations in a genotype  
            fitness_list: list that stores the dictionary which indicates the fitness of each epistic combinations 
                          in each site
            
        output: 
        -------
            float, the calculated fitness of a genotype (the genotype which corresponded to epistic_pair)
        '''
        
        one_fitness = []    
        
        for i in range(len(epistic_pair)):  # fitness_list[i] is the fitness dictionary that corresponds to site i.  
            one_fitness.append(fitness_list[i][epistic_pair[i]])  # epistic_pair[i] is the epistic combination in site i (the
                                                                # key in the dictionary)
        i_fitness = sum(one_fitness)/len(one_fitness)  # calculate the fitness
            
        return i_fitness
          
        
    @staticmethod
    def whether_fixed(fitness_1, fitness_2):
        
        s = (fitness_2/fitness_1)-1
        fix_pro = 2*s
        
        random_no = random.random(1)[0]
        
        if fix_pro > random_no:
            return True
        else:
            return False
    
    
    
    def adaptive_walk(self):
        '''Simulate one step  of adaptive walk along the landscape'''
        
        if self.start_type==0:  # self.start_type==0 (the initial given value), means initial step, randomly generates a 
                                # genotype as start_type (length is self.n)
            self.start_type = self.generate_start_type(self.n)
        else:  # start!='', not initial step, just pass
            pass
        
        neighbor_list = [self.start_type] # this list will be used to store the self.start_type and all its neighbors
                                        # (next step will select from the neighbors of self.start_type)
        
        neighbor_list += self.creat_neighbor(self.start_type, 1) # get all 1-mutant neighbors of start_type
                
        all_epistic_pair = [] # this list will be used to store all epistic combinations for genotypes in neighbor_list
        
        for i in neighbor_list:  # i is genotype in neighbor_list, self.epistic_site is network among the site
            all_epistic_pair.append(self.get_epistic_combination(i, self.epistic_site))
        
        
        allelic_set = []  # list that will be used to store all the epistic combinations in site 0, 1, 2, ..., self.n-1 
                        # (use set to exclude the repeat ones) for genotypes in neighbor_list
        
        for j in range(self.n): 
            pos_allelic_pair = [] # will be used to store all epistic combinations in site j for genotypes in neighbor_list
            for s in all_epistic_pair:
                pos_allelic_pair.append(s[j])   
            
            pos_allelic_set = list(set(pos_allelic_pair))  # convert the pos_allelic_pair to set then convert back to list to
                                                    # get the non-repeated epistic combinations in site j
                
            allelic_set.append(pos_allelic_set)  # append the non-repeated epistic combinations in site j into allelic_set to
                                        # get epistic combinations in all sites
            
        for s in range(self.n): # the length of allelic_set is self.n
            for i in allelic_set[s]: # i is the allelic combinations in site s, self.allelic_each_site is used to store the 
                                    #  episitc combinations at each site, self.allelic_each_site[s] is episitc combinations 
                                    # at site s that has already been generated
                if i not in self.allelic_each_site[s]: # if i is not in self.allelic_each_site[s], means epistic combination i
                                 # has not been generated, just append i into self.allelic_each_site[s] and generate a fitness  
                                 # for i and store into self.fitness_list[s](i will be the key)
                    self.allelic_each_site[s].append(i)
                    self.fitness_list[s][i] = random.random(1)[0]
            
        w_fitness = []   # will be used to store the calculated fitness of each genotype in neighbor_list
        
        for i in neighbor_list:
            if i not in self.genotype_list: # i not in self.genotype_list, means the fitness of genotype i has not been
                                            # calculated--> need to be calculated this time
                index_i = neighbor_list.index(i) # first get the index of i (this index is the same as the epistic 
                                                # combinations for genotype i in all_epistic_pair)
                i_fitness = self.cal_fitness(all_epistic_pair[index_i], self.fitness_list)
                w_fitness.append(i_fitness)
            else:  # otherwise it means that the fitness of genotype i has been calculated, just find it from the landscape
                w_fitness.append(self.landscape[i])
            
        neigh_fitness = dict(zip(neighbor_list[1:], w_fitness[1:]))  # creat a dictionary to store the genotypes and fitness of 
                                                            # the 1-mutant neighbors of self.start_type (start_type is not 
                                                            # included as it won't be selected for self.next_step) 
                                                            # key: genotype; value: fitness
    
        self.start_type_fitness = w_fitness[0]  # get the fitness of self.start_type
        
        possible_neigh = {k : v for k, v in neigh_fitness.items() if v >= self.start_type_fitness}  # get all possible   
                                                                    # neighbors for self.next_step (only if the fitness >= 
                                                                    # self.start_type fitness can be possible neighbors 
                                                                    # for self.next_step) 
        
        if possible_neigh:   # if possible_neigh is not empty, it means that possible genotypes for self.next_step exist;
                             # select the genotype for self.next_step according to their fitness
                
            pop_gens = list(possible_neigh.keys())   # get all the possible genotypes of self.next_step
            possible_next_type = pop_gens[random.randint(0, len(pop_gens))]  # find the possible_next_type and 
            possible_next_type_fit = possible_neigh[possible_next_type] # possible_next_type_fitness
            
            if self.whether_fixed(self.start_type_fitness, possible_next_type_fit): # check if the next_type can be fixed
                self.next_type = possible_next_type         # if can be fixed, then updated the self.next_type and its fitness
                self.next_type_fitness = possible_next_type_fit
            else:
                self.next_type = self.start_type
                self.next_type_fitness = self.start_type_fitness
                                                                                
        else: # if possible_neigh is empty, it means that there if no option for self.next_step-->cannot evolve
              #-->reach local optima
            self.next_type = ''
            self.next_type_fitness = 0   
           
    
        self.genotype_list.extend(neighbor_list)  # updating the self.genotype_list by storing all the genotype in neighbor_list
                                                # into self.genotype_list
            
        self.genotype_list = list(set(self.genotype_list))  # exclude the repeating genotypes in self.genotype_list
        
        landscape_subset = dict(zip(neighbor_list, w_fitness)) # the subset of landscape generated in this random_walk step
        
        self.landscape.update(landscape_subset)  # update self.landscape by the subset of landscape generated in this random
                                                 # _walk step
                
        
    def repeat_adaptive_walk(self):
        '''Simulate the random walk in the landscape'''
        
        self.walk_list = []  # this list will be used to store the steps of random walk
        
        fit = self.adaptive_walk()  # run the random_walk method
        
        self.walk_list.append([self.start_type, self.start_type_fitness])  # the initial step
        
        while self.next_type and self.next_type_fitness:  # this means that while next_step!='' and fitness[next_step]!=0.0,  
                                                         #not reach local optima
            self.walk_list.append([self.next_type, self.next_type_fitness]) # the next_step
            self.start_type = self.next_type  # update the self.start_type and self.start_type_fitness
            self.start_type_fitness = self.next_type_fitness
            fit = self.adaptive_walk() # run the random_walk method with the updated self.start_type 
        
        self.walk_genotype = list(set([i[0] for i in self.walk_list])) 
        self.new_walk_list = [[j, self.landscape[j]] for j in self.walk_genotype]
        self.new_walk_list.sort(key=lambda x: x[1])
        self.walk_genotype = [i[0] for i in self.new_walk_list]
 
        if len(self.new_walk_list) > 6:
            self.walk_genotype = self.walk_genotype[:6]
            self.new_walk_list = self.new_walk_list[:6]
            
    
    def search_different_allele(self):
        '''check all the alleles in each site for all genotypes involved in an adaptive walk(e.g. site 1: have '0' and '1',
        (means some genotypes have '0' in site 1 and some other have '1' in site 1),site 2: only have '0' (means all genotypes 
        involved in an adaptive walk have '0' in site 2))'''
        
#         self.repeat_adaptive_walk()  # first run the self.repeat_adaptive_walk() to get the self.walk_genotype
        
        walk_geno_list = []  
        for i in self.walk_genotype:   # first convert the genotypes into lists (each genotype will have a list) and store in
            walk_geno_list.append(list(i))  # walk_geno_list
            
        self.total_allele_list = []  # will be used to store the alleles present in each site for all genotypes involved in 
                                    # an adaptive walk
        for i in range(self.n): # for each site
            allele_list = []  # will be used to store all alleles in site i
            for j in walk_geno_list:  
                allele_list.append(j[i])
                
            allele_list = list(set(allele_list))#convert allele_list into a set to exclude the repeated one and then 
                                                # back to list
            self.total_allele_list.append(allele_list) # append the alleles in site i into self.total_allele_list
                
        
        self.shared_allele_indice = [] #will be used to store the indexes of shared alleles for the genotypes in an 
                                    # adaptive walk
        self.diff_allele_indice = [] #will be used to store the indexes of different alleles for the genotypes in 
                                    # an adaptive walk
        
        for i in range(self.n):
            if len(self.total_allele_list[i])>1: # this means site i has more than one alleles-->different allele,append i into
                self.diff_allele_indice.append(i) # self.diff_allele_indice
            else: # else means site i only have one allele for all genotypes--> shared allele, append i into self.shared_allele
                self.shared_allele_indice.append(i) # # _indice
                        
        self.shared_allele = ''  # # will be used to store the shared sequences for all genotypes
        
        for i in self.shared_allele_indice: 
            self.shared_allele+=self.walk_genotype[0][i] # # get the shared sequence by all genotype (use the first genotype in 
                                                      # adaptive walk to construct, also can use other genotypes in adatptive 
                                                      # walk because this sequence is shared by all genotypes
                    
        self.genotype_subset = []  # this list will store the subsets of genotypes in adaptive walk (here subset means that 
                         # only the different alleles are present, excluding the shared alleles for all genotypes in the walk)
        self.landscape_subset = {}  # this is what we would to get, the subset of landscape which only contain the information
                                    # of different alleles
        
        for i in self.walk_genotype: # get different allele sequence for each genotype in an adaptive walk
            s = '' 
            for j in self.diff_allele_indice: # the different allele sequence in genotype i
                s+=i[j]
            
            self.genotype_subset.append(s)  # append the subsets of genotypes into self.genotype_subet (will use the different
                                         # allele sequence to represent the whole genotype in an adaptive walk)
            self.landscape_subset[s] = self.landscape[i] # update the subset of landscape (key is the different allele 
                                                         # sequence in genotype i, value is the fitness of genotype i)
            
            
      
    def creat_landscape_subset(self):
        '''Generate the subset of lanscape which contains all possible combinations in different allele sites'''
        
#         self.search_different_allele()  # run the self.search_different_allele() first 
              
        all_possible_combination = [''.join(i) for i in itertools.product('01', repeat = len(self.diff_allele_indice))]
        # generate all possible allele combinations in different allele sites
        
        already_find = list(self.landscape_subset.keys()) # this is allele combinations that have already been got
        
        need_to_find_com = []  # will be used to store the allele combinations that haven't been got

        for i in all_possible_combination: 
             if i not in already_find:  # get all allele combinations that haven't been got
                need_to_find_com.append(i)

        need_to_find_com_list = []
        for i in need_to_find_com:   # convert each of the allele combinations in need_to_find_com to list and 
            need_to_find_com_list.append(list(i))  # store in need_to_find_com_list

            
        self.shared_allele_list = list(self.shared_allele) # convert the shared sequence (str) into a list

        orginal_genotype = []  # will be used to store the whole genotype constructed by the shared sequence and different 
                                # allele combinations (will be then used to find the fitness in self.landscape)
        for i in need_to_find_com_list: # only the different allele combinations that haven't been found (i.e. in need_to_find
                                        # _com_list)  will be converted back to the whole genotype sequence
            for j in self.shared_allele_indice:
                index_j = self.shared_allele_indice.index(j)
                i.insert(j, self.shared_allele_list[index_j]) # insert the alleles in shared_seq according to i according to the
            orginal_genotype.append(''.join(i)) # indices of shared_sequence in whole genotype sequence

            
        orginal_genotype_fitness = [] #will be used to store the fitness of each construced whole genotypes in orginal_genotype

        all_geno = list(self.landscape.keys()) # first get all whole genotype sequences in self.landscape

        for i in orginal_genotype:
            if i in all_geno:  # i in all_geno--> i already in self.landcape, get the fitness of i directly
                orginal_genotype_fitness.append(self.landscape[i])
            else:   # else i not in all_geno--> i not in self.landscape, need to calculate the fitness of i
                allelic_comb_i = self.get_epistic_combination(i, self.epistic_site) # generate the epistic pairs in i 
                                                                     # according to the network (i.e. self.epistic_site)
                    
                for j in range(self.n): # check whether the all_comb_i[j] have already been generated and given a fitness in 
                                        # site j
                    if allelic_comb_i[j] not in self.allelic_each_site[j]: # if not, just append the new combination into self.
                        self.allelic_each_site[j].append(allelic_comb_i[j]) # allelic_each_site[j] and given a fitness
                        self.fitness_list[j][allelic_comb_i[j]] = random.random(1)[0]

                fitness_i = self.cal_fitness(allelic_comb_i, self.fitness_list) # calcualte the fitness of genotype i

                orginal_genotype_fitness.append(fitness_i) # append the fitness of genotype i into orginal_genotype_fitness
                self.landscape[i] = fitness_i  # also update self.landscape with genotype i

        updated_landscape_subset = dict(zip(need_to_find_com, orginal_genotype_fitness)) # the updated_landscape_subset 
                                    # (need_to_find_com stores the keys, orginal_genotype_fitness stores the fitness,
                                    # need_to_find_com has the same order with orginal_genotype, orginal_genotype has the 
                                    # same order with orginal_genotype_fitness, so need_to_find_com has the same order with
                                    # with orginal_genotype_fitness)

        self.landscape_subset.update(updated_landscape_subset) # updated the self.landscape_subset with updated_landscape_subset.
    
    
    @staticmethod
    def compare_list(list1, list2):
        '''compare the elements in the two lists and return the indexes of different elements
        parameters: 
        ----------
            two lists with the same length (list1 and list2) to be compared
            
        output: 
        -------
            list that stores the indexes of the different elements in list1 and list2'''
        
        index_list = [] 
        
        for i in range(len(list1)):
            if list1[i] != list2[i]:
                index_list.append(i)
            else:
                pass
        
        return index_list
    
    
    def get_one_mutant(self):
        '''Get the one-site mutants (i.e.genotypes corresponding to W1, W2, W3, W4 and W5 in 
        Draghi 2013 paper Equation 5~8 )(the genotypes in adaptive walks are W12, W123, W1234 and W12345)'''
        
        self.ancestor = self.genotype_subset[0]  # first get the ancestor (the first genotype in an adaptive walk, self.
                         # genotype_subset uses the different allele sequences to represents the genotypes in adaptive walk)
            
        self.ancestor_fitness = self.landscape_subset[self.ancestor]
        
        ancestor_list = list(self.ancestor) # then convert the genotype of ancestor to list
        
        self.subset_change_index = []  # will be used to store the order of sites mutated in an adaptive walk
        
        for i in range(len(self.genotype_subset)-1): 
            change_index = self.compare_list(list(self.genotype_subset[i]),list(self.genotype_subset[i+1]))[0]
                 # compare the neighboring two genotypes in an adaptive walk to get the order of sites mutated
            self.subset_change_index.append(change_index) # append the sites mutated into self.subset_change_index
            
        self.one_mutant  = []  # will be used to store the one-site mutants (i.e. genotypes of W1, W2, W3, ...)
        
        for i in self.subset_change_index: # self.subset_change_index stores the indexes of mutated sites according to the 
                                            # mutated order
            start = ancestor_list.copy()
            if ancestor_list[i] =='0':
                start[i] = '1'
            else:
                start[i] = '0'
            
            self.one_mutant.append(''.join(start)) # get the one-site mutants
            
        self.one_mutant_fitness = []    
        for j in self.one_mutant: # get the fitness of the one_site mutants (i.e. W1, W2, W3, W4 and W5)
            self.one_mutant_fitness.append(self.landscape_subset[j])
        
        self.w = []
        
        for i in self.one_mutant_fitness:
            self.w.append(i/self.ancestor_fitness)
            
    @staticmethod
    def multiply_n_element(list1, n):
        '''Multiply the first n elements in list1
        parameters: 
        ----------
            list1: list
            n: int, the number of elements in list1 that would be multiplied
            
        output: 
        -------
            float, the multiplied results of the first n elements in list1'''
        
        m = 1
        
        for i in range(n):
            m*=list1[i]
            
        return m
        
    
            
    def calculate_epistatic_deviation(self):
        '''Calculate the epistatic deviations (i.e. e12, e123, e1234, e12345 in Draghi 2013 paper Equation 5~8)'''
        
        self.ep_dev = [] # will be used to store the epistatic deviations
        
        for i in range(2, len(self.diff_allele_indice)+1): # self.genotype_subset[2] is genotype corresponding to W12, 
                                # [3] is genotype corresponding to W123... self.multiply_n_element(self.one_mutant_fitness, 2)
                                # is W1*W2, self.multiply_n_element(self.one_mutant_fitness, 3) is  W1*W2*W3
            ep_dev = self.landscape_subset[self.genotype_subset[i]]/self.ancestor_fitness - self.multiply_n_element(self.w, i)
            self.ep_dev.append(ep_dev)
            
            
    def get_background_genotype(self):
           
        self.total_substitution_genotype = []
        self.total_background_genotype = []
        
        for i in self.subset_change_index:
            
            remain_comb_list = [list (j) for j in [''.join(s) for s in itertools.product('01', repeat = \
                                                                                         len(self.diff_allele_indice)-1)]]
            b_list = []
    
            for j in remain_comb_list:
                if self.ancestor[i] =='0':
                    j.insert(i, '1')   
                else:
                    j.insert(i, '0')
                    
                b_list.append(''.join(j))
            
            self.total_substitution_genotype.append(b_list)
            
            
        for i in self.subset_change_index:
            remain_comb_list = [list (j) for j in [''.join(s) for s in itertools.product('01', repeat = \
                                                                                         len(self.diff_allele_indice)-1)]]
            c_list = []
            for j in remain_comb_list:
                if self.ancestor[i] =='0':
                    j.insert(i, '0')   
                else:
                    j.insert(i, '1')
                
                c_list.append(''.join(j))
            
            self.total_background_genotype.append(c_list)
            
            
    def get_substitution_fitness(self):
        
        self.total_substitution_fitness = []
        self.total_background_fitness = []
        
        for i in self.total_substitution_genotype:
            one_site_fitness = []
            for j in i:
                one_site_fitness.append(self.landscape_subset[j])
            self.total_substitution_fitness.append(one_site_fitness)
            
        for i in self.total_background_genotype:
            one_site_fitness = []
            for j in i:
                one_site_fitness.append(self.landscape_subset[j])
            self.total_background_fitness.append(one_site_fitness)
            
    
    def calculate_fitness_effect(self):
        
        self.fitness_effect = []
        
        for i in range(len(self.total_background_fitness)):
            diff_fitness_list = []
            for j in range(len(self.total_background_fitness[0])):
                diff_fitness = self.total_substitution_fitness[i][j] - self.total_background_fitness[i][j]
                diff_fitness_list.append(diff_fitness)
                
            self.fitness_effect.append(diff_fitness_list)
            
            
    def calculate_spearman_coefficient(self):
        
        self.total_spearman_analyses = []
        self.total_spearman_coeff = []
        
        for i in range(len(self.total_background_fitness)):
            spearman_analyses = stats.spearmanr(self.fitness_effect[i], self.total_background_fitness[i])
            spearman_coeff = spearman_analyses[0]
            
            self.total_spearman_analyses.append(spearman_analyses)
            self.total_spearman_coeff.append(spearman_coeff)
            
            
    def find_shared_sequence(self):
        
        self.get_changed_site_index = []
        
        for i in range(1, len(self.subset_change_index)+1):
            one_site = []
            for j in range(i):
                one_site.append(self.subset_change_index[j])
                
            self.get_changed_site_index.append(one_site)
            
        
        all_index_list = list(range(len(self.ancestor)))
        
        self.total_same_allele_index = []
        
        for i in self.get_changed_site_index:
            same_allele_index = []
            for j in all_index_list:
                if j not in i:
                    same_allele_index.append(j)
            
            self.total_same_allele_index.append(same_allele_index)
        
        self.total_shared_seq = []
        
        for i in self.total_same_allele_index:
            shared_seq = []
            for j in i:
                shared_seq.append(self.ancestor[j])
                
            self.total_shared_seq.append(shared_seq)

    
    def construct_substitution_comb(self):
        
        self.whole_list = []
        
        a_list = self.total_shared_seq[1:]
        
        
        for i in a_list:
            all_comb = [list (j) for j in [''.join(s) for s in itertools.product('01', repeat = \
                                                                                         len(self.ancestor)-len(i))]]
            for j in i:
                for s in range(len(j)):
                    i.insert(self.subset_change_index[s], all_comb[s])
                    
            

In [49]:
a = NK_model(20, 7)
a.get_one_mutant()
a.calculate_epistatic_deviation()
a.get_background_genotype()
a.get_substitution_fitness()
a.calculate_fitness_effect()
a.calculate_spearman_coefficient()

print('1', a.new_walk_list, len(a.new_walk_list))
print('2', a.diff_allele_indice, len(a.diff_allele_indice))
print('3', a.genotype_subset)
print('3_1', a.subset_change_index)
print('4', a.landscape_subset, len(list(a.landscape_subset.keys())))
print('5', a.ancestor, a.ancestor_fitness)
print('6', a.one_mutant, a.one_mutant_fitness)
print('7', a.w)
print('8', a.ep_dev)
# print('9', a.one_site_background_list, len(a.one_site_background_list))
# print('10', a.all_remain_comb_list, len(a.all_remain_comb_list))
print('11', a.total_substitution_genotype, len(a.total_substitution_genotype[0][0]))
print('12', a.total_background_genotype, len(a.total_substitution_genotype[0]))
print('13', a.total_substitution_fitness)
print('14', a.total_background_fitness)
print('15', a.fitness_effect)
print('16', a.total_spearman_coeff)

a.find_shared_sequence()
print('17', a.get_changed_site_index)
print('18', a.total_same_allele_index)
print('19', a.total_shared_seq)

a.construct_substitution_comb()

1 [['10011010010100111101', 0.47725975103672524], ['10111010010100111101', 0.53793608365992929], ['10101010010100111101', 0.57140475014497727], ['10101011010100111101', 0.62107545218738358], ['10101011110100111101', 0.64430192180999224]] 5
2 [2, 3, 7, 8] 4
3 ['0100', '1100', '1000', '1010', '1011']
3_1 [0, 1, 2, 3]
4 {'1111': 0.55359924897129564, '0110': 0.47872404959606296, '0111': 0.60022187874128508, '0010': 0.49406868953513533, '0101': 0.50878697754592639, '1100': 0.53793608365992929, '1010': 0.62107545218738358, '1101': 0.53963385107412531, '1000': 0.57140475014497727, '0100': 0.47725975103672524, '1011': 0.64430192180999224, '0001': 0.57872509671616701, '1001': 0.61102073780254218, '0000': 0.50927964996359676, '1110': 0.54374591941044736, '0011': 0.62893965874211777} 16
5 0100 0.477259751037
6 ['1100', '0000', '0110', '0101'] [0.53793608365992929, 0.50927964996359676, 0.47872404959606296, 0.50878697754592639]
7 [1.1271348201716156, 1.0670911361314597, 1.0030681375417829, 1.066058

TypeError: list indices must be integers, not list

In [3]:
total_ep_dev = []

while len(total_ep_dev)<10:
    a = NK_model(20,7)
#     a.creat_landscape_subset()
    print('DIFF', a.diff_allele_indice, len(a.diff_allele_indice))
    print('WALK', a.walk_genotype, len(a.walk_genotype))
    
    if len(a.diff_allele_indice)==5:
        a.get_one_mutant()
        print('CHANGE', a.subset_change_index)
        a.calculate_epistatic_deviation()
        total_ep_dev.append(a.ep_dev)
        
print('TOTAL EP DEV', total_ep_dev, len(total_ep_dev))




DIFF [2, 10, 11, 14, 17] 5
WALK ['10110010010001010110', '10110010010101010110', '10010010010101010110', '10010010011101010110', '10010010011101010010', '10010010011101110010'] 6
CHANGE [2, 0, 1, 4, 3]
DIFF [2, 6, 7, 8] 4
WALK ['10111010101000110101', '10111010001000110101', '10111011001000110101', '10111001001000110101', '10011001001000110101'] 5
DIFF [4, 9, 18] 3
WALK ['01100010110100011011', '01100010110100011001', '01100010100100011001', '01101010100100011001'] 4
DIFF [2, 13, 14, 16, 18] 5
WALK ['01011111010101000110', '01011111010101000100', '01111111010101000100', '01111111010101001100', '01111111010100001100', '01111111010100101100'] 6
CHANGE [4, 0, 3, 1, 2]
DIFF [3, 6, 12, 16, 17] 5
WALK ['10001001100110011000', '10001001100110011100', '10001011100110011100', '10001011100100011100', '10001011100100010100', '10011011100100010100'] 6
CHANGE [4, 1, 2, 3, 0]
DIFF [4, 10, 13, 16, 18] 5
WALK ['11011000111001001010', '11010000111001001010', '11010000111001000010', '1101000011100000001

In [48]:
a = NK_model(20,7)
# a.repeat_adaptive_walk()


a.creat_landscape_subset()
print('GENO',a.walk_genotype)
print('1', a.walk_genotype)
print('2', a.shared_allele)
print('3', a.diff_allele_indice)
print('4', a.genotype_subset)
print('5', a.landscape_subset)

a.get_one_mutant()
print('6', a.subset_change_index)
print('7', a.one_mutant)
print('8', a.one_mutant_fitness)

a.calculate_epistatic_deviation()
print('9', a.ep_dev)

DIFF INDICE [0, 4, 5, 6, 14]
SHARED ALLE 000000100001001
GENO ['00001100001000001001', '00001100001000101001', '00001000001000101001', '00000000001000101001', '00000010001000101001', '10000010001000101001']
1 ['00001100001000001001', '00001100001000101001', '00001000001000101001', '00000000001000101001', '00000010001000101001', '10000010001000101001']
2 000000100001001
3 [0, 4, 5, 6, 14]
4 ['01100', '01101', '01001', '00001', '00011', '10011']
5 {'01100': 0.40851172831437382, '10111': 0.60801048411923142, '10110': 0.58131432647817516, '00000': 0.59893912440469643, '10000': 0.50877425796755538, '01110': 0.48915791992343471, '11000': 0.56930927503602369, '11001': 0.56035296488797715, '11010': 0.61883602190233533, '10100': 0.5039515475702292, '01010': 0.50675992164875816, '01011': 0.51458060127547989, '00101': 0.52810894113222506, '00010': 0.5616356236758423, '10101': 0.53522454586757928, '10010': 0.56932604152282118, '01111': 0.53760705101628048, '11101': 0.43837749619411681, '10011': 0.

In [32]:
total_subset = []

while len(total_subset)<10:
    a = NK_model(20, 7)
#     a.repeat_adaptive_walk()
#     a.creat_landscape_subset()
    
    if len(a.diff_allele_indice) ==5:
        total_subset.append(a.landscape_subset)
    
print('TOTAL SUBSET', total_subset, len(total_subset))

DIFF INDICE [6, 14]
SHARED ALLE 110001100100011010
DIFF INDICE [0, 1, 3, 7, 8]
SHARED ALLE 101101010010000
DIFF INDICE [5, 7, 10, 14, 18]
SHARED ALLE 101100111101010
DIFF INDICE [0, 7, 12, 14, 15]
SHARED ALLE 001011100011010
DIFF INDICE [0, 3, 6, 12]
SHARED ALLE 1110000000100000
DIFF INDICE [0, 1, 4, 6, 18]
SHARED ALLE 000100101100100
DIFF INDICE [0, 1, 8, 9, 14]
SHARED ALLE 010101010100011
DIFF INDICE [4]
SHARED ALLE 0001010001101111110
DIFF INDICE [0, 2, 14]
SHARED ALLE 01011110010011100
DIFF INDICE [0, 11, 16]
SHARED ALLE 00100011110011111
DIFF INDICE [0, 12, 17]
SHARED ALLE 11100011010100000
DIFF INDICE [7, 8, 9, 12, 14]
SHARED ALLE 101111101001111
DIFF INDICE [0, 4, 5, 10, 16]
SHARED ALLE 111111101100001
DIFF INDICE [2, 6, 8, 11, 13]
SHARED ALLE 100100100110111
DIFF INDICE [3, 4, 10, 18, 19]
SHARED ALLE 000001010101101
DIFF INDICE [0, 6, 8, 15, 17]
SHARED ALLE 110100010110010
TOTAL SUBSET [{'11111': 0.43768251884545262, '11010': 0.53379811388818199, '10100': 0.57727100493201589, '

In [22]:
a = NK_model(20,7)
# print(a.epistic_site)
# a.random_walk()
# a.repeat_adaptive_walk()
# # a.multiple_random_walk(3)
# # print('1', a.walk_list)
# # print('2', a.landscape)
# # print('LEN', len(list(a.landscape.keys())))
# # print('3', a.walk_genotype)
# a.search_different_allele()
# print('4', a.new_walk_list, len(a.new_walk_list))


# a.search_different_allele()

a.creat_landscape_subset()
print('5', a.landscape_subset, len(list(a.landscape_subset.keys())))



DIFF INDICE [0, 1, 3, 8, 9]
SHARED ALLE 100011110101110
5 {'01100': 0.55705782300521567, '10111': 0.52403817428686761, '10110': 0.57432678137506232, '00001': 0.60877277705176092, '00000': 0.5933255875072122, '10000': 0.5562102607858378, '01110': 0.60769328759747976, '11000': 0.56185577940186049, '11001': 0.56255940087866363, '11010': 0.54857885923788519, '10100': 0.54669404879719941, '01010': 0.63652907345767307, '01011': 0.6438557921555238, '11011': 0.54116200986799035, '00010': 0.65976707983258187, '10101': 0.55656319713163815, '10010': 0.60151724556107566, '01111': 0.57632738953464746, '11101': 0.55650152259939278, '10011': 0.59410039619118094, '01000': 0.62867148607151668, '00110': 0.63663848712065652, '11111': 0.46539259481540907, '00100': 0.52741911758917936, '11100': 0.54663237426495415, '11110': 0.51568120190360378, '01101': 0.58584968036501683, '00101': 0.55621097494898053, '10001': 0.55691388226264105, '01001': 0.64411867561606528, '00011': 0.6670937985304326, '00111': 0.6052